In [ ]:
import os
api_key = os.getenv("GCP_API_KEY")

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model = "gemini-3.5-flash-lite" , 
    temperature = 0 
) 

In [7]:
question = input("Ask someting")
res = llm.invoke(question)
res.content

d:\Agentic Ai\Monday-project\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'type': 'text',
  'text': 'Hi there! How can I help you today?',
  'extras': {'signature': 'El4KXAFpFH0TlNVpNHcMnzVOG1pjxqNe/58JG20tRCLClFCOH1HugOUZX6vXIZgWZ1eCsqFAzNasnJOrRVj19nXFrF/NxI2rG8a7rWr4FCAiK8lyKeS/kPwI5n6c3j11'}}]

In [8]:
import gradio as gr
from pypdf import PdfReader

reader = PdfReader("Resturaunt Q&A.pdf")
# print(reader)
restaurant_info = ""
for page in reader.pages:
    text = page.extract_text()
    if(text):
        restaurant_info+=text+'\n'
def chatbot(message , history):
    conversation  = ""
    for msg in history:
        role = msg["role"]
        content = msg["content"]

        if role == "user":
            conversation += f"Customer: {content}\n"

        elif role == "assistant":
            conversation += f"Assistant: {content}\n"
    prompt = f"""
You are restaurant customer support assistant.
Use the following restaurant information to 
answer the customer's question.
Restaurant Information:
{restaurant_info}
previous conversation : {conversation}
Customer Quesiton {message}
Answer fthe customer clearly and ploitely.
If the information is not available in the restaurat information , say that you dont f
have the information

"""
    response = llm.invoke(prompt)
    return response.content

demo = gr.ChatInterface(
    fn = chatbot , 
    title = "Restaurant support bot" , 
    description = "Ask questions about our restaurant"
)
demo.launch()

d:\Agentic Ai\Monday-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [9]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain_core.tools import tool

C:\Users\santh\AppData\Local\Temp\ipykernel_21936\3156583230.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [10]:
loader = PyPDFLoader("C:/Users/santh/Downloads/Santhosh_S_Resume.pdf")
documents = loader.load()

In [11]:
# Creating a simple tool
@tool
def read_pdf(question :str) ->str:
    """Answer a question using the pdf content """
    
    text = "\n".join(doc.page_content for doc in documents)
    return text


In [12]:
agent  = create_agent(
    model =llm ,
    tools = [read_pdf] , 
    system_prompt=""" You are a pdf reader agent .dist/Use the read_pdf tool to find information form the Answer only using
    information form the PDF. if the answer is not available in the pdf , say: ' I could not find this information in the pdf'"""
)

In [13]:
question = input("Ask someting about the pdf")
res = agent.invoke({
    "messages":[
        {"role":"user" , "content" : question}
    ]
})
answer = res["messages"][-1].content
if isinstance(answer, list):
    answer = "\n".join(
        item["text"]
        for item in answer
        if isinstance(item, dict) and item.get("type") == "text"
    )

answer

d:\Agentic Ai\Monday-project\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


'Hello! How can I help you with the PDF today?'

In [14]:
@tool
def solve(question : str) ->str:
  """Solve the queston"""
  return question

In [15]:
agent =  create_agent(
    model = llm ,
    tools = [solve],  
    system_prompt="""
You are a mathematical agent.

Your job is to solve mathematical expressions.

Valid mathematical operations include:
+ addition
- subtraction
* multiplication
/ division
% modulus
^ exponentiation

Examples of valid inputs:
10 + 20
50 - 15
5 * 6
20 / 4
10 % 3
2 ^ 3

If the user's input contains a mathematical expression, solve it.

If the input is not a mathematical expression, respond exactly:
"This is not a mathematical operation."
"""
)

In [16]:
question = input("Enter any mathematical operation")
res = agent.invoke({
    "messages":[{"role":"user" , "content" :question} ]
})
answer = res["messages"][-1].content
question
print(answer)


d:\Agentic Ai\Monday-project\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
d:\Agentic Ai\Monday-project\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'This is not a mathematical operation.', 'extras': {'signature': 'El4KXAFpFH0TJPyBFnpAskCIoRW0xrX2ZwIcueNLhTFvTp92CL9qFnJVd5cD+HXjHfnbPkxATqichqZi/B2iktsCca96FWUUI4iZK7zlfIxNAwtRZkAbHM4CxaFVU6sr'}}]


In [17]:
import wikipedia
res = wikipedia.summary("who is rashmika")
res

'Rashmika Mandanna (born 5 April 1996) is an Indian actress who primarily works in Telugu and Hindi films. Her accolades include four SIIMA Awards and a Filmfare Award South. One of India\'s highest-paid actresses, she was featured in Forbes India\'s 2024 list of "30 Under 30".\nAfter a brief modelling career in 2014, Mandanna made her acting debut with the Kannada romantic comedy Kirik Party (2016) and gained further commercial success with the action film Anjani Putra and the romantic drama Chamak (both 2017). She expanded into Telugu cinema in 2018 with the comedy drama Chalo and achieved her breakthrough with the romantic comedy Geetha Govindam, which earned her the Filmfare Critics Award for Best Actress – Telugu. She went on to star as the leading lady in the action comedies Sarileru Neekevvaru and Bheeshma (both 2020).\nFurther recognition came with the pan-India success of the Telugu action film Pushpa: The Rise (2021) and a supporting role in the period drama Sita Ramam (2022)